## Sample Usage
### Basic

In [ ]:
from ghIssueAnalyzer.agents import IssueAnalyzer
import os
from dotenv import load_dotenv
from ghIssueAnalyzer.models import ChatGPT
import pandas as pd

# Make sure to have a .env file with OPENAI_API_KEY and GITHUB_TOKEN in the project root
load_dotenv()

gpt = ChatGPT(os.getenv('OPENAI_API_KEY'))

agent = IssueAnalyzer(
  'Automattic/mongoose',
  os.getenv('GITHUB_TOKEN'),
  gpt)

analysis = agent.fetch_issues().analyze(head=15, template='DX')

pd.DataFrame(analysis).to_csv('issue_analysis.csv', index=False)
print('Results exported to issue_analysis.csv')

### Listening to progress update events

In [ ]:
from ghIssueAnalyzer.agents import IssueAnalyzer
import os
from dotenv import load_dotenv
from ghIssueAnalyzer.models import ChatGPT
import logging
import pandas as pd
from pydispatch import dispatcher

logging.basicConfig(level=logging.INFO)
load_dotenv()

gpt = ChatGPT(os.getenv('OPENAI_API_KEY'))
agent = IssueAnalyzer(
  'Automattic/mongoose',
  os.getenv('GITHUB_TOKEN'),
  gpt)

# Define callback functions for progress, error, and finish signals
def on_progress(step=None, data=None):
    print(f'Step {step}: {data}')
    
def on_error(data=None):
    print(f'Received error event: {data}')
    
def on_finish(data=None):  
  pd.DataFrame(data).to_csv('issue_analysis.csv', index=False)
  print('Results exported to issue_analysis.csv')

# Connect the callback functions to the agent's signals
dispatcher.connect(
  on_progress,
  signal=IssueAnalyzer.Signals.PROGRESS_UPDATE,
  sender=agent)
  
dispatcher.connect(
  on_finish,
  signal=IssueAnalyzer.Signals.TASK_COMPLETED,
  sender=agent)

dispatcher.connect(
  on_error,
  signal=IssueAnalyzer.Signals.ERROR,
  sender=agent)

# Start the analysis
analysis = agent.fetch_issues().analyze(head=15)